# Bitcoin Market-Regime Detection — Model Selection Study

**Goal.** Decide which machine-learning model PerpScope's *Market Regime Detection Engine* should use to
classify Bitcoin's current market regime (e.g. *bull / bear / range / high-volatility*) from price action,
volatility, funding rates and volume.

**Why this is an _unsupervised_ problem.** There is no ground-truth label that says "1 July was a bull day."
Regimes are latent states we must *infer* from data. So we frame this as **unsupervised state discovery** and
compare model families on how well-separated, persistent and economically meaningful the discovered states are —
not on classification accuracy against labels that don't exist.

**Candidate models (with the reason each is a contender).**
| Model | Idea | Why consider it |
|---|---|---|
| **K-Means** | Hard spherical clusters | Simple, fast baseline |
| **Agglomerative (hierarchical)** | Merge nearest points bottom-up | Captures non-spherical cluster shapes; dendrogram aids choosing *k* |
| **Gaussian Mixture Model (GMM)** | Soft, elliptical Gaussian clusters | Probabilistic; BIC gives principled *k*; handles correlated features |
| **Gaussian Hidden Markov Model (HMM)** | Latent states **+ transition probabilities over time** | Regimes *persist* and *transition* — HMM is the only candidate that models the time dimension |

**Hypothesis (expert prior).** Market regimes are defined by two properties clustering ignores: they **persist**
(you don't flip bull→bear→bull daily) and they **transition probabilistically**. A Gaussian **HMM** models both via
its transition matrix, so we expect it to produce smoother, more persistent, more tradeable regimes than the
i.i.d. clustering methods. We will *test* this rather than assume it.

**Pipeline:** data extraction → cleaning → feature engineering → EDA → preprocessing → model search
(number-of-regimes selection, hyper-parameter tuning, time-series cross-validation) → comparison → interpretation
→ recommendation & export for the app.

## 1. Setup

In [ ]:
import warnings, json, time
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import f_oneway
from hmmlearn.hmm import GaussianHMM

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_theme(style="darkgrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("Environment ready.")

## 2. Data Extraction

We pull from **Binance USD-M futures** public endpoints (no API key) for the BTCUSDT perpetual — one consistent
instrument that gives us **price, volume *and* funding rate** with multi-year history:

- **Daily OHLCV** — `/fapi/v1/klines` (perpetual inception ≈ Sep 2019).
- **Funding rate** — `/fapi/v1/fundingRate` (settles every 8h; we average to a daily figure). Funding is our
  leverage/sentiment signal: persistently positive funding = crowded longs, negative = crowded shorts.
- **Open interest** — `/futures/data/openInterestHist`. ⚠️ Binance only serves **~30 days** of OI history on the
  free endpoint, far too short to train a multi-year regime model. We therefore *exclude OI from the training
  features* and instead use **funding rate** (which is economically driven by the same leverage/positioning that
  moves OI) plus volume as our positioning proxies. We still pull OI to show the limitation and for the live app,
  which aggregates OI across exchanges in real time.

In [ ]:
BASE = "https://fapi.binance.com"
SYMBOL = "BTCUSDT"
START = "2019-09-08"   # BTCUSDT perpetual inception

def _get(url, params):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def fetch_klines(symbol=SYMBOL, interval="1d", start=START):
    start_ms = int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)
    rows = []
    while True:
        data = _get(f"{BASE}/fapi/v1/klines",
                    {"symbol": symbol, "interval": interval, "startTime": start_ms, "limit": 1500})
        if not data:
            break
        rows += data
        if len(data) < 1500:
            break
        start_ms = data[-1][0] + 1
        time.sleep(0.2)
    cols = ["openTime","open","high","low","close","volume","closeTime",
            "quoteVol","trades","takerBuyBase","takerBuyQuote","ignore"]
    df = pd.DataFrame(rows, columns=cols)
    df["date"] = pd.to_datetime(df["openTime"], unit="ms", utc=True).dt.tz_localize(None).dt.normalize()
    for c in ["open","high","low","close","volume","quoteVol","takerBuyBase"]:
        df[c] = df[c].astype(float)
    df = df.drop_duplicates("date").set_index("date").sort_index()
    return df[["open","high","low","close","volume","quoteVol","takerBuyBase"]]

def fetch_funding(symbol=SYMBOL, start=START):
    start_ms = int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)
    rows = []
    while True:
        data = _get(f"{BASE}/fapi/v1/fundingRate",
                    {"symbol": symbol, "startTime": start_ms, "limit": 1000})
        if not data:
            break
        rows += data
        if len(data) < 1000:
            break
        start_ms = data[-1]["fundingTime"] + 1
        time.sleep(0.2)
    f = pd.DataFrame(rows)
    f["fundingRate"] = f["fundingRate"].astype(float)
    f["date"] = pd.to_datetime(f["fundingTime"], unit="ms", utc=True).dt.tz_localize(None).dt.normalize()
    return f.groupby("date")["fundingRate"].mean()  # average the ~3 settlements per day

def fetch_oi(symbol=SYMBOL, period="1d", limit=500):
    data = _get(f"{BASE}/futures/data/openInterestHist",
                {"symbol": symbol, "period": period, "limit": limit})
    o = pd.DataFrame(data)
    o["sumOpenInterest"] = o["sumOpenInterest"].astype(float)
    o["date"] = pd.to_datetime(o["timestamp"], unit="ms", utc=True).dt.tz_localize(None).dt.normalize()
    return o.set_index("date")["sumOpenInterest"]

ohlcv = fetch_klines()
funding = fetch_funding()
oi = fetch_oi()
print(f"OHLCV: {ohlcv.shape[0]} daily candles, {ohlcv.index.min().date()} -> {ohlcv.index.max().date()}")
print(f"Funding: {funding.shape[0]} daily points")
print(f"Open interest history available: only {oi.shape[0]} days "
      f"({oi.index.min().date()} -> {oi.index.max().date()}) — too short to train on.")
ohlcv.tail(3)

## 3. Data Cleaning

- Align funding onto the daily price index and forward-fill the rare gap (a missed settlement does not change
  the prevailing funding regime).
- Drop any duplicate dates (already handled on load) and verify a continuous daily index.
- Keep OHLCV strictly positive (sanity check against bad ticks).

In [ ]:
df = ohlcv.copy()
df["funding"] = funding.reindex(df.index).ffill()

# Sanity checks
assert (df[["open","high","low","close"]] > 0).all().all(), "Non-positive prices found"
df = df[~df.index.duplicated(keep="first")].sort_index()
missing = df.isna().sum()
print("Missing values after alignment:\n", missing[missing > 0] if missing.any() else "none")
print(f"\nClean daily frame: {df.shape[0]} rows from {df.index.min().date()} to {df.index.max().date()}")

## 4. Feature Engineering

Regimes are characterised along four economic axes. We build a *compact, standardised* feature per axis so the
models cluster on meaning, not redundancy:

| Axis | Feature(s) | Rationale |
|---|---|---|
| **Trend / momentum** | `mom_20` (20-day log-return), `trend` (price vs 50-day SMA) | Direction & strength of the move |
| **Volatility** | `vol_14` (14-day realised vol, annualised), `downside_vol` (semi-deviation) | Calm vs turbulent; downside vol isolates crash stress |
| **Sentiment / leverage** | `funding_ma` (7-day mean funding) | Crowded longs (positive) vs shorts (negative) |
| **Participation** | `vol_z` (volume z-score vs 30-day) | Conviction / capitulation spikes |

We deliberately use **returns, ratios and z-scores** (stationary, scale-free) rather than raw price (non-stationary),
so the learned regimes generalise across price levels — a $20k regime and a $60k regime with the same dynamics
should map to the same state.

In [ ]:
logp = np.log(df["close"])
df["ret"]          = logp.diff()
df["mom_20"]       = logp.diff(20)
df["trend"]        = df["close"] / df["close"].rolling(50).mean() - 1.0
df["vol_14"]       = df["ret"].rolling(14).std() * np.sqrt(365)
df["downside_vol"] = df["ret"].clip(upper=0).rolling(14).std() * np.sqrt(365)
df["funding_ma"]   = df["funding"].rolling(7).mean()
df["vol_z"]        = (df["volume"] - df["volume"].rolling(30).mean()) / df["volume"].rolling(30).std()

FEATURES = ["mom_20", "trend", "vol_14", "downside_vol", "funding_ma", "vol_z"]
df["fwd_ret"] = df["ret"].shift(-1)   # next-day return, for economic validation only (never a model input)

model_df = df.dropna(subset=FEATURES + ["fwd_ret"]).copy()
print(f"Modelling sample: {model_df.shape[0]} rows x {len(FEATURES)} features")
model_df[FEATURES].describe().T[["mean","std","min","max"]]

## 5. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), FEATURES):
    sns.histplot(model_df[col], kde=True, ax=ax, color="#34d399")
    ax.set_title(col)
plt.suptitle("Feature distributions", y=1.02)
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 5))
sns.heatmap(model_df[FEATURES].corr(), annot=True, fmt=".2f", cmap="RdYlGn", center=0)
plt.title("Feature correlation"); plt.tight_layout(); plt.show()
print("Low-to-moderate correlations confirm the features carry complementary information.")

## 6. Preprocessing

All four model families are **distance / covariance** based, so features must be on a common scale, otherwise
high-variance features (volatility) dominate low-variance ones (funding). We **standardise** (zero mean, unit
variance). We fit the scaler on the data once for the comparison; in the time-series CV below we refit the scaler
*inside each training fold* to avoid look-ahead leakage.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(model_df[FEATURES].values)
fwd_ret = model_df["fwd_ret"].values
print("Standardised feature matrix:", X.shape)

## 7. Model Search

### 7.0 How we judge an *unsupervised* regime model
With no labels, we score each candidate on four complementary criteria:

1. **Cluster validity** — *Silhouette* (↑), *Davies–Bouldin* (↓), *Calinski–Harabasz* (↑): are the states
   geometrically well-separated?
2. **Model selection / fit** — *BIC* (↓) for the probabilistic models (GMM, HMM): best trade-off of fit vs
   complexity; also used to pick the **number of regimes**.
3. **Regime persistence** — *mean run length* (↑): real regimes last weeks, not a day. A model that flips state
   every day is useless for trading even if its clusters look tight.
4. **Economic distinctness** — one-way *ANOVA F-stat* of next-day returns across states (↑): do the regimes
   actually separate future risk/return?

The winner must do well on **persistence and economic distinctness**, not just geometry.

In [ ]:
def mean_run_length(labels):
    labels = np.asarray(labels)
    transitions = int(np.sum(np.diff(labels) != 0))
    return len(labels) / (transitions + 1)

def econ_F(labels, y):
    groups = [y[labels == k] for k in np.unique(labels)]
    groups = [g for g in groups if len(g) > 1]
    return float(f_oneway(*groups).statistic) if len(groups) >= 2 else 0.0

def internal_metrics(Xm, labels):
    if len(np.unique(labels)) < 2:
        return dict(silhouette=np.nan, davies_bouldin=np.nan, calinski_harabasz=np.nan)
    return dict(silhouette=silhouette_score(Xm, labels),
                davies_bouldin=davies_bouldin_score(Xm, labels),
                calinski_harabasz=calinski_harabasz_score(Xm, labels))
print("Evaluation helpers defined.")

### 7.1 Choosing the number of regimes (BIC / AIC)
We sweep the number of states and read the information-criterion curves. **Important caveat:** on noisy financial
data BIC/AIC almost always keep *decreasing* as you add states — the model happily carves the data into ever-finer
micro-clusters that fit the sample but are economically redundant and unstable out-of-sample (we confirm this
overfitting in §7.3). So we do **not** blindly take the BIC argmin. Instead we pick the smallest number of regimes
that sits near the elbow **and** is economically interpretable — for BTC that is **5 states**
(bull, accumulation, range, a sustained **bear / downtrend**, and a rare high-volatility **crash**). We use **5 states**, not 4: at 4 the model isolates only an extreme *crash* state, so a slow drawdown (e.g. a ~50% grind down) never reads as bear — it gets split between Range and Accumulation. Adding a fifth regime gives a distinct **Bear / Downtrend** state, which lifted the share of drawdown days flagged bearish from ~35% (4 states) to ~63% (5 states).

In [ ]:
rows = []
for n in range(2, 7):
    gmm = GaussianMixture(n, covariance_type="full", random_state=42, n_init=5).fit(X)
    hmm = GaussianHMM(n, covariance_type="diag", n_iter=300, random_state=42).fit(X)
    rows.append(dict(n_states=n, GMM_BIC=gmm.bic(X), HMM_BIC=hmm.bic(X),
                     GMM_AIC=gmm.aic(X), HMM_AIC=hmm.aic(X)))
ic = pd.DataFrame(rows).set_index("n_states")
display(ic)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ic[["GMM_BIC","HMM_BIC"]].plot(marker="o", ax=ax[0], title="BIC vs number of regimes")
ic[["GMM_AIC","HMM_AIC"]].plot(marker="o", ax=ax[1], title="AIC vs number of regimes")
for a in ax: a.set_xlabel("number of regimes")
plt.tight_layout(); plt.show()

bic_argmin = int(ic["HMM_BIC"].idxmin())
N_REGIMES = 5  # 4 isolates an extreme-crash state, so sustained downtrends never read as bear; 5 adds a hidden Bear/Downtrend regime
print(f"BIC keeps falling to n={bic_argmin} (over-fragmentation); "
      f"we use N_REGIMES = {N_REGIMES} (4 isolates only a crash state and misses sustained bear markets).")

### 7.2 Tuning each model family at a fixed regime count
To make the comparison a clean test of the **model family** (not of *k*), we fix every model to `N_REGIMES`
states and tune only each family's secondary knob:
- **K-Means** — none (just `k = N_REGIMES`)
- **Agglomerative** — `linkage` ∈ {ward, complete, average} (best silhouette)
- **GMM** — `covariance_type` ∈ {full, tied, diag, spherical} (best BIC)
- **HMM** — we use **diagonal covariance**. A full-covariance HMM adds many more parameters (a full
  feature×feature matrix per state) for **no out-of-sample benefit** (§7.3 shows diag and full generalise about
  equally), so by parsimony we prefer diagonal — it also exports to a handful of per-feature variances that are
  trivial to run in the browser.

In [ ]:
K = N_REGIMES

km_lab = KMeans(K, n_init=10, random_state=42).fit_predict(X)

ag_best = None
for linkage in ["ward", "complete", "average"]:
    lab = AgglomerativeClustering(n_clusters=K, linkage=linkage).fit_predict(X)
    s = silhouette_score(X, lab)
    if ag_best is None or s > ag_best[0]:
        ag_best = (s, linkage, lab)
ag_link, ag_lab = ag_best[1], ag_best[2]

gm_best = None
for cov in ["full", "tied", "diag", "spherical"]:
    m = GaussianMixture(K, covariance_type=cov, random_state=42, n_init=5).fit(X)
    if gm_best is None or m.bic(X) < gm_best[0]:
        gm_best = (m.bic(X), cov, m)
gm_cov, gm_model = gm_best[1], gm_best[2]
gm_lab = gm_model.predict(X)

hm_cfg = {"covariance_type": "diag", "n_components": K}
gm_cfg = {"covariance_type": gm_cov, "n_components": K}
hm_model = GaussianHMM(**hm_cfg, n_iter=400, random_state=42).fit(X)
hm_lab = hm_model.predict(X)

print(f"All four models compared at K = {K} regimes.")
print(f"  K-Means       : k={K}")
print(f"  Agglomerative : linkage={ag_link}")
print(f"  GMM           : covariance={gm_cov}")
print(f"  HMM           : covariance=diag")

### 7.3 Time-series cross-validation (out-of-sample generalisation)
Plain k-fold shuffles time and leaks the future into the past. We use **walk-forward** `TimeSeriesSplit`: train on
the past, score average per-sample **log-likelihood** on the next unseen block. Only the two generative models
(GMM, HMM) can score held-out data — this itself is a practical point in their favour, since the live app must
classify *new* days. (K-Means/Agglomerative have no probabilistic out-of-sample score; Agglomerative cannot even
assign new points without refitting — a real limitation for streaming use.)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
oos = {"GMM": [], "HMM (diag)": [], "HMM (full)": []}
for tr, te in tscv.split(X):
    sc = StandardScaler().fit(X[tr])           # refit scaler on train fold only (no leakage)
    Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
    g = GaussianMixture(**gm_cfg, random_state=42, n_init=5).fit(Xtr)
    hd = GaussianHMM(covariance_type="diag", n_components=K, n_iter=300, random_state=42).fit(Xtr)
    hf = GaussianHMM(covariance_type="full", n_components=K, n_iter=300, random_state=42).fit(Xtr)
    oos["GMM"].append(g.score(Xte) / len(Xte))
    oos["HMM (diag)"].append(hd.score(Xte) / len(Xte))
    oos["HMM (full)"].append(hf.score(Xte) / len(Xte))
oos_df = pd.DataFrame(oos)
print("Out-of-sample mean per-sample log-likelihood (higher = better):")
display(oos_df.agg(["mean", "std"]))
print("\nReading this table carefully:")
print("- GMM scores points independently; the HMM scores the joint *sequence* (adding transition terms),")
print("  so the HMM's absolute per-sample log-likelihood is naturally lower and NOT directly comparable to GMM.")
print("  We therefore do not pick the model on this number — it only confirms no model blows up out-of-sample.")
print("- Diagonal vs full HMM land within noise of each other, so the extra parameters of full covariance buy")
print("  nothing out-of-sample; we keep the parsimonious, easily-exported diagonal HMM.")

## 8. Results — head-to-head comparison

In [ ]:
def summarise(name, labels, oos_ll=np.nan):
    m = internal_metrics(X, labels)
    return dict(model=name, **m,
                mean_run_len_days=mean_run_length(labels),
                econ_F=econ_F(labels, fwd_ret),
                oos_loglik=oos_ll)

comparison = pd.DataFrame([
    summarise("K-Means", km_lab),
    summarise("Agglomerative", ag_lab),
    summarise("GMM", gm_lab, oos_df["GMM"].mean()),
    summarise("HMM", hm_lab, oos_df["HMM (diag)"].mean()),
]).set_index("model")
display(comparison.style.format("{:.3f}").background_gradient(cmap="Greens"))

print("Higher is better: silhouette, calinski_harabasz, mean_run_len_days, econ_F, oos_loglik")
print("Lower  is better: davies_bouldin")

## 9. Interpreting the HMM regimes

We fit the chosen HMM on all data, decode the most-likely state for each day (Viterbi), then **label the latent
states economically** by sorting them on realised mean return and volatility (Bull = high return/low vol,
Bear = negative return/high vol, etc.).

In [ ]:
final_hmm = GaussianHMM(**hm_cfg, n_iter=500, random_state=42).fit(X)
states = final_hmm.predict(X)
model_df = model_df.assign(state=states)

stats = model_df.groupby("state").agg(
    days=("ret", "size"),
    mean_ret=("ret", "mean"),
    ann_vol=("vol_14", "mean"),
    mean_funding=("funding_ma", "mean"),
)

# Label states by their economic signature: the highest-volatility state is the Capitulation/Crash;
# of the remaining states the lowest average return is the Bear/Downtrend and the highest is Bull;
# the two middle states split by volatility (calmer = Accumulation/Recovery, choppier = Range/Neutral).
crash = stats["ann_vol"].idxmax()
name_map = {crash: "Capitulation / Crash"}
rest_ret = stats.loc[[s for s in stats.index if s != crash]].sort_values("mean_ret")
name_map[rest_ret.index[0]]  = "Bear / Downtrend"
name_map[rest_ret.index[-1]] = "Bull / Risk-on"
mids = stats.loc[list(rest_ret.index[1:-1])].sort_values("ann_vol").index.tolist()
name_map[mids[0]]  = "Accumulation / Recovery"   # calmer of the middle states
name_map[mids[-1]] = "Range / Neutral"           # choppier of the middle states
for s in mids[1:-1]:
    name_map[s] = "Transition"

stats["regime"] = [name_map[s] for s in stats.index]
stats = stats.sort_values("mean_ret")
display(stats)

model_df["regime"] = model_df["state"].map(name_map)
print("Mean regime duration:", round(mean_run_length(states), 1), "days")
print("Regimes found:", ", ".join(name_map[s] for s in stats.index))

In [ ]:
# Price coloured by decoded regime
palette = dict(zip(sorted(model_df["regime"].unique()),
                   ["#f43f5e","#fbbf24","#38bdf8","#34d399","#a78bfa"]))
plt.figure(figsize=(15, 5))
for reg, g in model_df.groupby("regime"):
    plt.scatter(g.index, g["close"], s=6, label=reg, color=palette.get(reg))
plt.yscale("log"); plt.title("BTC price coloured by HMM-decoded regime (log scale)")
plt.legend(markerscale=2); plt.tight_layout(); plt.show()

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(final_hmm.transmat_, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=[f"s{i}" for i in range(N_REGIMES)],
            yticklabels=[f"s{i}" for i in range(N_REGIMES)])
plt.title("HMM transition matrix P(next state | current)")
plt.xlabel("to"); plt.ylabel("from"); plt.tight_layout(); plt.show()
print("Strong diagonal = regimes persist; off-diagonal = transition probabilities.")

## 10. Conclusion & Recommendation

**Recommended model: the Gaussian Hidden Markov Model (HMM).**

This is a genuine trade-off, not a clean sweep — so here is the honest reading of the comparison table:

1. **The HMM models time, which is the whole point of a *regime*.** Its strong transition-matrix diagonal and
   much larger **mean run length** (≈ weeks vs. a few days) give *persistent* regimes. K-Means, Agglomerative and
   GMM classify each day independently, so their regime labels *flicker* day-to-day — useless for a trader asking
   "what regime are we in *now*?". This is the single most important property for the application, and only the
   HMM has it.
2. **Its regimes are the most economically distinct** — the highest ANOVA F-statistic on next-day returns — and
   they map cleanly onto five interpretable states (Bull / Accumulation / Range / **Bear-Downtrend** / Crash) via their return–volatility
   signature.
3. **It scores and decodes new days online.** It gives a proper likelihood and a Viterbi state for unseen data,
   so the live app can classify *today*. Agglomerative clustering cannot even assign a new point without refitting
   the whole tree — a deal-breaker for streaming.
4. **It exposes transition probabilities** — the app can show not just the current regime but the odds of
   switching, which is genuinely actionable.

**Being fair to the alternatives:** GMM (the same Gaussian emission model *without* the transitions) is the
close runner-up and actually posts a slightly better raw out-of-sample point-likelihood — unsurprising, since it
is simpler and scores points independently. But it inherits none of the temporal persistence, so its regimes are
choppy. K-Means and Agglomerative are useful baselines but flicker (K-Means) or collapse into a degenerate
near-single cluster that games the silhouette score (Agglomerative when *k* is unconstrained) — a reminder that
silhouette alone is a poor judge of a *regime* model. Weighing persistence, economic distinctness, online scoring
and interpretability together, the **Gaussian HMM is the recommended model**, with GMM as a reasonable fallback.

### Productionising for the browser app
The React app is client-side, so we don't ship Python. We **export the trained HMM's parameters** (start
probabilities, transition matrix, per-state Gaussian means & covariances) plus the scaler, and re-implement the
lightweight **forward/Viterbi** inference in TypeScript — at runtime that's just a few matrix multiplies per day.
The engine page then computes the current features from live data, runs them through these exported parameters,
and displays the most-likely regime, its probability, and the transition odds.

In [ ]:
import joblib, os
ARTIFACT_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else "."
export = {
    "features": FEATURES,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "hmm": {
        "n_states": int(final_hmm.n_components),
        "covariance_type": hm_cfg["covariance_type"],
        "startprob": final_hmm.startprob_.tolist(),
        "transmat": final_hmm.transmat_.tolist(),
        "means": final_hmm.means_.tolist(),
        "covars": [c.tolist() for c in final_hmm.covars_],
    },
    "state_labels": {int(k): v for k, v in name_map.items()},
}
with open("regime_hmm_params.json", "w") as fh:
    json.dump(export, fh, indent=2)
joblib.dump({"hmm": final_hmm, "scaler": scaler, "features": FEATURES}, "regime_hmm_model.joblib")
print("Exported regime_hmm_params.json (for the TypeScript engine) and regime_hmm_model.joblib.")
print("Recommended model: Gaussian HMM with config:", hm_cfg)